# Exercise 13 - Interpolation

In this exercise, we load some basic temperature data from New York City from the end of 2018 and the start of 2019. We then simulate a simple recurring equipment failure at 3:00 and 6:00 a.m. preventing us from getting temperature readings at those hours. How well does interpolation help us, and how far off are the interpolated mean and median calculations from the original, true values?

1.  Load the temperature data from New York City (from the ```nyc-temps.txt``` file) into a series. The measurements are in degrees Celsius.

In [159]:
import numpy as np
import pandas as pd
from numpy import nan

temperature = pd.read_csv('./nyc-temps.txt').squeeze()

print(f"temperature preview:\n{temperature.head()}\nnumber of rows: {temperature.shape[0]}")


temperature preview:
0   -1
1   -1
2   -1
3   -1
4   -1
Name: -1, dtype: int64
number of rows: 728


2. Create a data frame with two columns: temp, with the temperatures, and hour, representing the hours at which the measurements were taken. The hour values should be 0, 3, 6, 9, 12, 15, 18, and 21, repeated for all 728 data points.

In [160]:
temp_df = pd.DataFrame(
  {
      "hour": [0, 3, 6, 9, 12, 15, 18, 21] * 91,
      "temperature": temperature
  })

temp_df.head(5)

,hour,temperature
0,0,-1
1,3,-1
2,6,-1
3,9,-1
4,12,-1


3. Calculate the mean and median values. These are the real values, which we hope to replicate via interpolation.

In [161]:
temp_mean = temp_df["temperature"].mean()
temp_median = temp_df["temperature"].median()

print(f"temperature mean: {temp_mean:.2f} °C\ntemperature median: {temp_median:.2f} °C")

temperature mean: -1.05 °C
temperature median: 0.00 °C


4. Set all values from 3:00 and 6:00 a.m. to ```NaN```.

In [162]:
nan_temp = temp_df.copy()
nan_temp.loc[temp_df["hour"].isin([3, 6]), "temperature"] = nan

nan_temp.head(8)

,hour,temperature
0,0,-1.0
1,3,NaN
2,6,NaN
3,9,-1.0
4,12,-1.0
5,15,-1.0
6,18,-1.0
7,21,1.0


5.  Interpolate the values with the ```interpolate``` method.

In [163]:
linear_interpolation = nan_temp.interpolate(method='linear')

linear_interpolation.head()

,hour,temperature
0,0,-1.0
1,3,-1.0
2,6,-1.0
3,9,-1.0
4,12,-1.0


6. What are the mean and median of the interpolated data frame? Are they similar to the real values? Why or why not?

In [164]:
print(f"mean temperature: {linear_interpolation['temperature'].mean():.2f} °C\n"
      f"median temperature: {linear_interpolation['temperature'].median():.2f} °C")

mean temperature: -1.05 °C
median temperature: 0.00 °C


## Beyond the exercise

* How does the behavior of ```interpolate``` change if you use ```method='nearest'```?

In [165]:
nearest_interpolation = nan_temp.interpolate(method='nearest')

nearest_interpolation.head()

,hour,temperature
0,0,-1.0
1,3,-1.0
2,6,-1.0
3,9,-1.0
4,12,-1.0


In [166]:
print(f"mean temperature: {nearest_interpolation['temperature'].mean():.2f} °C\n"
      f"median temperature: {nearest_interpolation['temperature'].median():.2f} °C")

mean temperature: -1.05 °C
median temperature: 0.00 °C


* Let’s assume the equipment works fine around the clock but fails to record readings at –1 degrees and below. Are the interpolated values similar to the real (missing) values they replace? Why or why not?

In [167]:
nan_below_minus_1 = temp_df.copy()

nan_below_minus_1.loc[nan_below_minus_1["temperature"] <= -1, "temperature"] = nan

nan_below_minus_1.head(8)

,hour,temperature
0,0,NaN
1,3,NaN
2,6,NaN
3,9,NaN
4,12,NaN
5,15,NaN
6,18,NaN
7,21,1.0


In [168]:
linear_interpolation = nan_below_minus_1.interpolate(method='linear')
linear_interpolation.head(8)

,hour,temperature
0,0,NaN
1,3,NaN
2,6,NaN
3,9,NaN
4,12,NaN
5,15,NaN
6,18,NaN
7,21,1.0


In [174]:
print(f"mean temperature: {linear_interpolation['temperature'].mean():.2f} °C\n"
      f"median temperature: {linear_interpolation['temperature'].median():.2f} °C\n")

mean temperature: 2.02 °C
median temperature: 1.00 °C



In [170]:
nearest_interpolation = nan_below_minus_1.interpolate(method='nearest')

nearest_interpolation.head(8)

,hour,temperature
0,0,NaN
1,3,NaN
2,6,NaN
3,9,NaN
4,12,NaN
5,15,NaN
6,18,NaN
7,21,1.0


In [171]:
print(f"mean temperature: {nearest_interpolation['temperature'].mean():.2f} °C\n"
      f"median temperature: {nearest_interpolation['temperature'].median():.2f} °C\n")

mean temperature: 2.02 °C
median temperature: 1.00 °C



From above it can be seen that not all ```NaN``` were interpolated. Also, the mean and median temperatures differ from the original case.

* A cheap solution to interpolation is to replace ```NaN``` values with the column’s mean. Do this (with the missing values from –1 and below), and compare the new mean and median. Again, why are (or aren’t) these values similar to the original ones?

In [172]:
temp_mean = nan_below_minus_1["temperature"].mean()
nan_below_minus_1.loc[np.isnan(nan_below_minus_1["temperature"]), "temperature"] = round(temp_mean, 2)

nan_below_minus_1.head(16)


,hour,temperature
0,0,2.76
1,3,2.76
2,6,2.76
3,9,2.76
4,12,2.76
5,15,2.76
6,18,2.76
7,21,1.00
8,0,1.00
9,3,1.00


In [173]:
print(f"mean temperature: {nan_below_minus_1['temperature'].mean():.2f} °C\n"
      f"median temperature: {nan_below_minus_1['temperature'].median():.2f}° C\n")

mean temperature: 2.76 °C
median temperature: 2.76° C



Applying this solution is even worse than trying to interpolate ```NaN``` values.